# m05 lab · Time–frequency tools: spectra, the analytic signal and wavelets

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Dominant frequency, organization index and phase mapping (m29) all rest on a few tools: the discrete Fourier
transform, windows, spectral averaging, the analytic signal and wavelets. Here you test each one on signals whose
answer you know, and see where electrogram-like signals break the assumptions.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Frequency resolution, and why zero padding does not improve it | eq-m05-resolution |
| B | Windows and leakage: a weak tone next to a strong one | eq-m05-leakage |
| C | The variance of a spectrum, and Welch averaging | eq-m05-welch |
| D | Pulse trains: why a raw electrogram's spectral peak is a harmonic | eq-m05-pulse-train |
| E | The analytic signal and instantaneous phase | eq-m05-analytic |
| F | Time–frequency trade-off and the Morlet wavelet | eq-m05-uncertainty, eq-m05-morlet |

All signals are synthetic, in relative units. Each part ends with **checks**. No downloads; an Octave copy of parts A–E
is in `labs/octave/m05_lab.m`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import timefreq as F
rng = np.random.default_rng(5)

## Part A · Frequency resolution

The DFT of $N$ samples at $f_s$ has bins $f_s/N = 1/T$ apart (eq-m05-resolution). Two tones closer than about $1/T$
merge into one peak. Zero padding evaluates the same spectrum on a finer grid; it does not separate them.

In [ ]:
fs = 200.0
res = {}
fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for k, T in enumerate((0.5, 1.0, 2.0)):
    t, x = F.two_tone(fs, T, 6.0, 7.0)
    f, P = F.periodogram(x, fs, "boxcar", nfft=64 * len(x))
    res[T] = F.resolved(f, P, 6.0, 7.0)
    ax[k].plot(f, P / P.max()); ax[k].set_xlim(2, 11); ax[k].set_title(f"T = {T} s: {'two peaks' if res[T] else 'one peak'}")
    ax[k].set_xlabel("frequency (Hz)")
plt.show()
t, x = F.two_tone(fs, 0.5, 6.0, 7.0)
f, P = F.periodogram(x, fs, "boxcar", nfft=1024 * len(x))
padded_helps = F.resolved(f, P, 6.0, 7.0)
print("resolved:", res, "| 0.5 s with 1024x zero padding:", padded_helps)

In [ ]:
# Checks
assert not res[0.5] and res[1.0] and res[2.0]
assert not padded_helps

## Part B · Windows and leakage

A finite record multiplies the signal by a window, which smears each spectral line into the window's own spectrum
(eq-m05-leakage). The rectangular window has the narrowest main lobe but sidelobes only about 13 dB down; the Hann
window's main lobe is twice as wide and its sidelobes are about 31 dB down. Put a tone 40 dB weaker next to a
strong one, off the bin grid, 15 Hz away, and look for it.

In [ ]:
rect_sl, rect_null = F.window_sidelobe_db("boxcar")
hann_sl, hann_null = F.window_sidelobe_db("hann")
print(f"rectangular: highest sidelobe {rect_sl:.1f} dB, first null {rect_null:.0f} bin; Hann: {hann_sl:.1f} dB, {hann_null:.0f} bins")
t, x = F.two_tone(fs, 2.0, 10.25, 25.25, a2=0.01)
seen = {}
for win in ("boxcar", "hann"):
    f, P = F.periodogram(x, fs, win, nfft=16 * len(x))
    db = 10 * np.log10(P / P.max())
    near = (f > 24.5) & (f < 26.0)
    between = (f > 18.0) & (f < 22.0)
    seen[win] = db[near].max() - db[between].max()       # how far the weak tone stands above the floor nearby
    plt.plot(f, db, label=win)
plt.xlim(0, 30); plt.ylim(-90, 5); plt.xlabel("frequency (Hz)"); plt.ylabel("dB"); plt.legend(); plt.show()
print({k: round(v, 1) for k, v in seen.items()}, "dB above the leakage floor")

In [ ]:
# Checks
assert abs(rect_sl + 13.26) < 0.1 and abs(hann_sl + 31.47) < 0.1
assert seen["hann"] > 30 and seen["boxcar"] < 3          # Hann reveals it; rectangular leakage hides it

## Part C · The variance of a spectrum

The periodogram of noise does not get smoother with a longer record: at every bin its standard deviation equals its
mean, because each bin is estimated from one complex number. Averaging $K$ independent segments (Welch's method)
divides the spread by $\sqrt K$ but costs resolution, since each segment is $K$ times shorter (eq-m05-welch). In one
technical study of dominant-frequency analysis, records longer than 2 s were needed for reliable estimates (Ng et al.
2007).

In [ ]:
cv = {K: F.periodogram_cv(segments=K, trials=200) for K in (1, 4, 8)}
cv_long = F.periodogram_cv(T=32.0, segments=1, trials=50)
for K, v in cv.items():
    print(f"K = {K}: std / mean = {v:.3f} (1/sqrt(K) = {1 / np.sqrt(K):.3f})")
print(f"one 32-s periodogram: std / mean = {cv_long:.3f}")

In [ ]:
# Checks
assert abs(cv[1] - 1) < 0.03 and abs(cv_long - 1) < 0.04
assert abs(cv[8] * np.sqrt(8) - 1) < 0.05

## Part D · Pulse trains and the dominant frequency

A train of identical pulses every $T_0$ has spectral lines at $k/T_0$, weighted by the pulse's own spectrum $P(f)$
(eq-m05-pulse-train). A sharp, zero-mean biphasic deflection of width $w$ has $P(0) = 0$ and $\lvert P\rvert$ largest
near $1/(2\pi w)$, so the largest line of a raw train sits at a harmonic far above the activation rate. Band-pass,
rectify and low-pass turns each deflection into one smooth bump, and the peak moves to $1/T_0$. Preprocessing is
needed for raw bipolar electrograms (Ng et al. 2007); m29 defines the dominant frequency on top of it.

In [ ]:
fs, cycle, width = 1000.0, 0.18, 0.004
t, x, times = F.pulse_train(fs, 8.0, cycle, width)
raw = F.peak_frequency(x, fs, 2.0, 400.0)
pre = F.rectify_lowpass(x, fs)
rect = F.peak_frequency(pre, fs, 3.0, 15.0)
print(f"activation rate 1/T0 = {1 / cycle:.2f} Hz; raw spectral peak {raw:.2f} Hz (harmonic {raw * cycle:.1f}); "
      f"pulse spectrum peaks at 1/(2 pi w) = {1 / (2 * np.pi * width):.1f} Hz; after preprocessing {rect:.2f} Hz")
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
f, P = F.periodogram(x - x.mean(), fs, "hann", nfft=8 * len(x)); ax[0].plot(f, P / P.max()); ax[0].set_xlim(0, 120); ax[0].set_title("raw train")
f, P = F.periodogram(pre - pre.mean(), fs, "hann", nfft=8 * len(x)); ax[1].plot(f, P / P.max()); ax[1].set_xlim(0, 30); ax[1].set_title("band-pass, rectify, low-pass")
for a in ax: a.set_xlabel("frequency (Hz)")
plt.show()

In [ ]:
# Checks
assert round(raw * cycle) > 3 and abs(raw - round(raw * cycle) / cycle) < 0.2
assert abs(rect - 1 / cycle) < 0.1

## Part E · The analytic signal and instantaneous phase

The analytic signal $z = x + j\mathcal H[x]$ turns a real signal into a rotating vector (eq-m05-analytic). For a
sinusoid, $\mathcal H[\cos] = \sin$ and the phase turns once per period; for a carrier whose envelope varies slowly,
$\lvert z\rvert$ recovers the envelope. Electrogram-like signals break the narrowband assumption. Count phase turns
against activations for one deflection per cycle, two deflections per cycle, and one deflection plus noise. Phase from
contact electrograms depends on deflection shape and noise (Kuklik et al. 2014), which is why m29 rebuilds the
signal before taking its phase.

In [ ]:
fs = 1000.0
t = np.arange(0, 4, 1 / fs)
x = np.cos(2 * np.pi * 5 * t)
env = 1 + 0.5 * np.cos(2 * np.pi * 0.5 * t)
am = env * np.cos(2 * np.pi * 20 * t)
env_err = np.max(np.abs(np.abs(F.analytic(am)) - env)[200:-200])
print(f"sinusoid: {F.phase_cycles(x):.2f} turns for 20 periods; AM envelope recovered to {env_err:.1e}")
w = 0.004
g = lambda u: np.exp(-0.5 * (u / w) ** 2)
times = np.arange(0.09, 4, 0.18)
single = sum(-(t - tk) / w * g(t - tk) for tk in times)
double = single + sum(-0.8 * (t - tk - 0.04) / w * g(t - tk - 0.04) for tk in times)
noisy = single + 0.15 * rng.standard_normal(len(t))
turns = {name: F.phase_cycles(s) for name, s in (("single", single), ("double", double), ("noisy", noisy))}
print(f"{len(times)} activations; phase turns:", {k: round(v, 1) for k, v in turns.items()})

In [ ]:
# Checks
assert abs(F.phase_cycles(x) - 20) < 0.02 and env_err < 1e-3
assert abs(turns["single"] - len(times)) < 0.5
assert abs(turns["double"] - 2 * len(times)) < 0.5
assert turns["noisy"] > 3 * len(times)

## Part F · Time and frequency together

A signal cannot be both short and narrowband: the RMS duration and RMS bandwidth satisfy
$\sigma_t\sigma_f \ge 1/(4\pi)$, with equality only for a Gaussian (eq-m05-uncertainty). Wavelets use that bound
well: the Morlet wavelet is a Gaussian-windowed complex sinusoid, and at scale $s$ it is centered on
$f = \omega_0/(2\pi s)$ (eq-m05-morlet). Track a preprocessed train whose cycle length shortens from 200 to 150 ms.

In [ ]:
xs = np.linspace(-20, 20, 40001)
st, sf = F.spreads(xs, np.exp(-xs ** 2 / 2))
st2, sf2 = F.spreads(xs, np.where(np.abs(xs) < 1, 1 - xs ** 2, 0.0))
print(f"Gaussian: sigma_t sigma_f = {st * sf:.5f} (1/4pi = {1 / (4 * np.pi):.5f}); a parabolic pulse: {st2 * sf2:.5f}")

fs = 1000.0
t = np.arange(0, 8, 1 / fs)
cl = 0.2 - 0.05 * t / 8                                   # cycle length shortens from 200 to 150 ms
phase = np.cumsum(1 / cl) / fs                            # activations at integer phase
acts = np.searchsorted(phase, np.arange(1, int(phase[-1]))) / fs
x = sum(F.biphasic_pulse(t - tk, 0.004) for tk in acts)
pre = F.rectify_lowpass(x, fs)
freqs = np.linspace(3, 9, 61)
W = np.abs(F.cwt(pre - pre.mean(), fs, freqs))
ridge = freqs[np.argmax(W, axis=0)]
early, late = np.median(ridge[1000:2000]), np.median(ridge[6000:7000])
print(f"wavelet ridge: {early:.2f} Hz early (true about {1 / np.interp(1.5, t, cl):.2f}), {late:.2f} Hz late (true about {1 / np.interp(6.5, t, cl):.2f})")
plt.pcolormesh(t[::10], freqs, W[:, ::10], shading="auto"); plt.plot(t, 1 / cl, "w--", lw=1)
plt.xlabel("time (s)"); plt.ylabel("frequency (Hz)"); plt.title("Morlet scalogram of the preprocessed train"); plt.show()

In [ ]:
# Checks
assert abs(st * sf - 1 / (4 * np.pi)) < 1e-4 and st2 * sf2 > 1 / (4 * np.pi)
assert abs(early - 1 / np.interp(1.5, t, cl)) < 0.3 and abs(late - 1 / np.interp(6.5, t, cl)) < 0.3

## Where to go next

- In part A, try a Hann window at T = 1 s. Are 6 and 7 Hz still resolved? Why not?
- In part C, use 50% overlapping segments. How much does the spread fall?
- In part E, add the double potential with a 20-ms gap instead of 40 ms. What happens to the phase?
- m29 builds dominant frequency, organization index and phase maps on these tools.